In [4]:
from __future__ import annotations

import asyncio
import json
import re
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional
from urllib.parse import parse_qs, urljoin, urlsplit

import requests
from playwright.async_api import async_playwright
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# If Playwright cannot find Chromium, run: playwright install chromium
TARGETS = [
    {"name": "ZDavP-2", "url": "http://pisrs.si/Pis.web/pregledPredpisa?id=ZAKO4703"},
    {"name": "Pravilnik ZDavP-2", "url": "http://pisrs.si/Pis.web/pregledPredpisa?id=PRAV7927"},
    {"name": "ZDDV-1", "url": "http://pisrs.si/Pis.web/pregledPredpisa?id=ZAKO4701"},
    {"name": "Pravilnik ZDDV-1", "url": "http://pisrs.si/Pis.web/pregledPredpisa?id=PRAV7542"},
    {"name": "ZDoh-2", "url": "http://pisrs.si/Pis.web/pregledPredpisa?id=ZAKO4697"},
    {"name": "ZDDPO-2", "url": "http://pisrs.si/Pis.web/pregledPredpisa?id=ZAKO4687"},
    {"name": "Pravilnik ZDDPO-2", "url": "http://pisrs.si/Pis.web/pregledPredpisa?id=PRAV8299"},
]

OUTPUT_DIR = Path("downloads/pisrs")
HEADLESS = True
NAVIGATION_TIMEOUT_MS = 120_000
MAX_ATTEMPTS_PER_DOCUMENT = 3


In [5]:
def slugify(value: str) -> str:
    normalized = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    cleaned = re.sub(r"[^a-zA-Z0-9]+", "-", normalized).strip("-").lower()
    return cleaned or "document"


def parse_npb_number(text: str | None) -> Optional[int]:
    if not text:
        return None
    match = re.search(r"\bNPB\s*(\d+)\b", text, re.IGNORECASE)
    return int(match.group(1)) if match else None


def build_requests_session() -> requests.Session:
    session = requests.Session()
    retries = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update(
        {
            "User-Agent": (
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            )
        }
    )
    return session


async def ensure_latest_npb(page) -> Dict[str, Any]:
    info: Dict[str, Any] = {
        "dropdown_present": False,
        "current_label": None,
        "current_npb": None,
        "selected_npb": None,
        "available_npbs": [],
    }
    dropdown_locator = page.locator('[data-test="results-text-npb-dropdown"]')
    if await dropdown_locator.count() == 0:
        return info

    dropdown = dropdown_locator.first
    info["dropdown_present"] = True
    current_label = (await dropdown.inner_text()).strip()
    current_npb = parse_npb_number(current_label)
    info["current_label"] = current_label
    info["current_npb"] = current_npb

    await dropdown.click()
    await page.wait_for_selector('[role="option"]', state="visible", timeout=10_000)
    options = page.locator('[role="option"]')

    labels: List[str] = []
    npb_values: List[int] = []
    for i in range(await options.count()):
        label = (await options.nth(i).inner_text()).strip()
        if not label:
            continue
        labels.append(label)
        parsed = parse_npb_number(label)
        if parsed is not None:
            npb_values.append(parsed)

    if not npb_values:
        await page.keyboard.press("Escape")
        info["selected_npb"] = current_npb
        return info

    highest_npb = max(npb_values)
    info["available_npbs"] = sorted(set(npb_values), reverse=True)

    if current_npb != highest_npb:
        target_label = next(label for label in labels if parse_npb_number(label) == highest_npb)
        await options.filter(has_text=target_label).first.click()
        await page.wait_for_timeout(1_000)
    else:
        await page.keyboard.press("Escape")

    refreshed_label = (await dropdown.inner_text()).strip()
    info["selected_npb"] = parse_npb_number(refreshed_label) or highest_npb
    return info


async def extract_html_download_url(page) -> str:
    html_button = page.locator('div.files [data-test="result-oblike-html"]')
    if await html_button.count() == 0:
        raise RuntimeError("HTML button not found in div.files")

    anchor = html_button.first.locator("xpath=ancestor::a[1]")
    href = await anchor.get_attribute("href")
    if not href:
        raise RuntimeError("HTML download URL was empty")

    return urljoin(page.url, href)


def download_html(session: requests.Session, file_url: str, output_path: Path) -> int:
    response = session.get(file_url, timeout=120)
    response.raise_for_status()
    content_type = (response.headers.get("Content-Type") or "").lower()
    if "html" not in content_type:
        raise RuntimeError(f"Unexpected content type for {file_url}: {content_type}")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_bytes(response.content)
    return len(response.content)


async def scrape_pisrs_documents(
    targets: List[Dict[str, str]],
    output_dir: Path,
    *,
    headless: bool = True,
    nav_timeout_ms: int = 120_000,
    max_attempts: int = 3,
) -> List[Dict[str, Any]]:
    output_dir.mkdir(parents=True, exist_ok=True)
    session = build_requests_session()
    results: List[Dict[str, Any]] = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        context = await browser.new_context(locale="sl-SI")
        page = await context.new_page()

        for target in targets:
            law_id = parse_qs(urlsplit(target["url"]).query).get("id", ["UNKNOWN"])[0]

            for attempt in range(1, max_attempts + 1):
                try:
                    await page.goto(target["url"], wait_until="networkidle", timeout=nav_timeout_ms)
                    await page.wait_for_selector("div.files", state="attached", timeout=30_000)

                    npb_info = await ensure_latest_npb(page)
                    html_url = await extract_html_download_url(page)

                    selected_npb = npb_info.get("selected_npb")
                    npb_suffix = f"_NPB{selected_npb}" if selected_npb is not None else ""
                    file_name = f"{law_id}_{slugify(target['name'])}{npb_suffix}.html"
                    file_path = output_dir / file_name
                    file_size = await asyncio.to_thread(download_html, session, html_url, file_path)

                    results.append(
                        {
                            "name": target["name"],
                            "law_id": law_id,
                            "source_url": target["url"],
                            "selected_npb": selected_npb,
                            "available_npbs": npb_info.get("available_npbs"),
                            "html_download_url": html_url,
                            "file_path": str(file_path),
                            "file_size_bytes": file_size,
                            "attempt": attempt,
                            "status": "ok",
                        }
                    )
                    break
                except Exception as exc:
                    if attempt == max_attempts:
                        results.append(
                            {
                                "name": target["name"],
                                "law_id": law_id,
                                "source_url": target["url"],
                                "attempt": attempt,
                                "status": "error",
                                "error": f"{type(exc).__name__}: {exc}",
                            }
                        )
                    else:
                        await asyncio.sleep(attempt)

        await context.close()
        await browser.close()

    report_path = output_dir / "download_report.json"
    report_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    session.close()
    return results


In [6]:
results = await scrape_pisrs_documents(
    TARGETS,
    OUTPUT_DIR,
    headless=HEADLESS,
    nav_timeout_ms=NAVIGATION_TIMEOUT_MS,
    max_attempts=MAX_ATTEMPTS_PER_DOCUMENT,
)

for item in results:
    if item["status"] == "ok":
        print(f"OK | {item['name']} | NPB {item.get('selected_npb')} | {item['file_path']}")
    else:
        print(f"ERROR | {item['name']} | {item['error']}")

print(f"\\nSaved report: {OUTPUT_DIR / 'download_report.json'}")


OK | ZDavP-2 | NPB 31 | downloads/pisrs/ZAKO4703_zdavp-2_NPB31.html
OK | Pravilnik ZDavP-2 | NPB 25 | downloads/pisrs/PRAV7927_pravilnik-zdavp-2_NPB25.html
OK | ZDDV-1 | NPB 20 | downloads/pisrs/ZAKO4701_zddv-1_NPB20.html
OK | Pravilnik ZDDV-1 | NPB 30 | downloads/pisrs/PRAV7542_pravilnik-zddv-1_NPB30.html
OK | ZDoh-2 | NPB 36 | downloads/pisrs/ZAKO4697_zdoh-2_NPB36.html
OK | ZDDPO-2 | NPB 23 | downloads/pisrs/ZAKO4687_zddpo-2_NPB23.html
OK | Pravilnik ZDDPO-2 | NPB 2 | downloads/pisrs/PRAV8299_pravilnik-zddpo-2_NPB2.html
\nSaved report: downloads/pisrs/download_report.json
